# Test an OpenVINO Policy Without a Physical Robot

Use the PhysicalAI inference runtime to load an exported Pi0.5 policy, evaluate it in the official LIBERO benchmark, and display a rollout recorded from LIBERO's MuJoCo environment.

<img src="media/4.Simulation-only.png" width="800">

## 1) Verify Notebook Dependencies

Complete the shared environment setup and the [optional Notebook 004 LIBERO dependencies](README.md#optional-notebook-004-libero-dependencies), then launch this notebook with the **PhysicalAI Tutorials (uv)** kernel.

This cell only verifies the prepared environment; it does not modify packages while the Jupyter kernel is running.

In [ ]:
from importlib.metadata import version

import ipywidgets
from physicalai.benchmark.gyms import LiberoBenchmark
from physicalai.inference import InferenceModel

if not hasattr(InferenceModel, "from_pretrained"):
    raise RuntimeError(
        "The installed PhysicalAI runtime does not provide InferenceModel.from_pretrained(). "
        "Repeat the notebook 004 dependency setup from README.md, then restart this kernel."
    )

print(f"PhysicalAI Studio: {version('physicalai-train')}")
print(f"ipywidgets: {ipywidgets.__version__}")
print("Notebook dependencies are ready.")

## 2) Configure the Model and Benchmark

The default path downloads the released OpenVINO policy package from Hugging Face with `InferenceModel.from_pretrained()`. If accessing Hugging Face from the PRC fails because of a connection issue, set `HF_ENDPOINT=https://hf-mirror.com` before launching JupyterLab as described in [README.md](README.md).

Set `MODEL_SOURCE = "local"` to evaluate your own OpenVINO export instead. `LOCAL_MODEL_DIR` must point to the exported policy package directory, not to a training checkpoint. A compatible local model must use the observation and action schema expected by the selected LIBERO task suite.

`LiberoBenchmark` creates the LIBERO environment and downloads its required assets on first use. `RECORD_MODE = "all"` guarantees that the smoke-test rollout is saved for display.

In [ ]:
from pathlib import Path
import os
import time

# MuJoCo chooses its renderer when it is first imported.
os.environ.setdefault("MUJOCO_GL", "egl")

import ipywidgets as widgets
import openvino as ov
from IPython.display import Markdown, Video, display
from physicalai.benchmark.gyms import LiberoBenchmark
from physicalai.inference import InferenceModel

# Model source: "huggingface" (recommended) or "local".
MODEL_SOURCE = "huggingface"
MODEL_REPO_ID = "OpenVINO/pi05-libero-fp16-ov"
MODEL_REVISION = "main"
MODEL_CACHE_DIR = Path("physicalai_assets/models")
LOCAL_MODEL_DIR = Path("/path/to/local/openvino-export").expanduser()

# Keep the default run small; increase these values for a full evaluation.
TASK_SUITE = "libero_10"
TASK_IDS = [0]
NUM_EPISODES = 1
MAX_STEPS = None
SEED = 42

OUTPUT_DIR = Path("physicalai_outputs/004")
VIDEO_DIR = OUTPUT_DIR / "videos"
RESULTS_PATH = OUTPUT_DIR / "benchmark_results.json"
RECORD_MODE = "all"

MODEL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
VIDEO_DIR.mkdir(parents=True, exist_ok=True)

## 3) Select an OpenVINO Device

The selector lists the devices reported by the installed OpenVINO runtime and prefers an available GPU. Select CPU when you only need to verify the workflow.

In [ ]:
core = ov.Core()
device_options = list(core.available_devices)
if not device_options:
    raise RuntimeError("OpenVINO did not report any available inference devices.")

default_device = next(
    (device for device in device_options if device == "GPU" or device.startswith("GPU.")),
    "CPU" if "CPU" in device_options else device_options[0],
)
TARGET_DEVICE = widgets.Dropdown(
    options=device_options,
    value=default_device,
    description="Device:",
)
display(TARGET_DEVICE)

## 4) Load the OpenVINO Policy

`InferenceModel.from_pretrained()` downloads and loads a Hugging Face policy package. For a local PhysicalAI/OpenVINO export, instantiate `InferenceModel` with the export directory. Both paths use the current inference API and the selected OpenVINO device.

In [ ]:
selected_device = TARGET_DEVICE.value

if MODEL_SOURCE == "huggingface":
    policy = InferenceModel.from_pretrained(
        MODEL_REPO_ID,
        revision=MODEL_REVISION,
        cache_dir=MODEL_CACHE_DIR,
        backend="openvino",
        device=selected_device,
    )
elif MODEL_SOURCE == "local":
    if not LOCAL_MODEL_DIR.is_dir():
        raise FileNotFoundError(
            f"Local OpenVINO policy package not found: {LOCAL_MODEL_DIR}"
        )
    policy = InferenceModel(
        LOCAL_MODEL_DIR,
        backend="openvino",
        device=selected_device,
    )
else:
    raise ValueError("MODEL_SOURCE must be 'huggingface' or 'local'.")

print(f"Policy loaded on {selected_device} from {MODEL_SOURCE}.")

## 5) Run the Official LIBERO Benchmark

This is a real policy rollout in LIBERO's MuJoCo simulation, not a scripted or approximate arm animation. `LiberoBenchmark` sends environment observations to the OpenVINO `InferenceModel`, applies the predicted actions, computes benchmark metrics, and records the rollout through Physical AI Studio's `VideoRecorder`.

The default configuration runs one episode of one task as a functional smoke test. Increase `TASK_IDS` and `NUM_EPISODES` for meaningful model evaluation.

In [ ]:
benchmark_started_at = time.time()
benchmark = LiberoBenchmark(
    task_suite=TASK_SUITE,
    task_ids=TASK_IDS,
    num_episodes=NUM_EPISODES,
    max_steps=MAX_STEPS,
    seed=SEED,
    video_dir=VIDEO_DIR,
    record_mode=RECORD_MODE,
)
results = benchmark.evaluate(policy)

print(results.summary())
results.to_json(RESULTS_PATH)
print(f"Saved benchmark results to: {RESULTS_PATH.resolve()}")

## 6) Display the Recorded MuJoCo Rollout

This video is **not generated by the policy model** and is **not a replay converted from the training dataset**. It records a closed-loop evaluation in the LIBERO simulation:

1. LIBERO resets a task and MuJoCo renders the simulated camera observations.
2. `InferenceModel` runs the OpenVINO policy on those observations and predicts a robot action.
3. LIBERO applies that action to the simulated robot, and MuJoCo computes the resulting motion, contacts, and object state.
4. The updated observation is sent back to the policy, repeating until the episode ends.
5. Physical AI Studio's `VideoRecorder` saves the rendered frames as an MP4 file.

The benchmark stores these MP4 files below `VIDEO_DIR`, grouped by policy and task. Therefore, the video visualizes the behavior produced by OpenVINO inference inside the MuJoCo environment. The following cell displays the videos written or updated by the most recent benchmark run.

In [ ]:
recent_videos = sorted(
    (
        path
        for path in VIDEO_DIR.rglob("*.mp4")
        if path.stat().st_mtime >= benchmark_started_at - 2
    ),
    key=lambda path: path.stat().st_mtime,
)

if not recent_videos:
    recent_videos = sorted(
        VIDEO_DIR.rglob("*.mp4"),
        key=lambda path: path.stat().st_mtime,
    )[-1:]

if not recent_videos:
    raise FileNotFoundError(
        f"No benchmark video was created below {VIDEO_DIR.resolve()}."
    )

for video_path in recent_videos:
    display(Markdown(f"**{video_path.relative_to(VIDEO_DIR)}**"))
    display(Video(filename=str(video_path), embed=True, html_attributes="controls loop"))

## 7) Interpret the Result

The summary and JSON file contain the success rate, reward, episode length, and rollout performance reported by the benchmark. The embedded MP4 is the corresponding LIBERO/MuJoCo episode driven by the OpenVINO policy.

A one-episode run confirms that the complete deployment pipeline executes. It is not statistically meaningful; use more tasks, episodes, and seeds before comparing models or devices.